# JULES TEAM Environment & Core Capabilities
This notebook sets up the environment, clones the Jules repository, vectorizes it using Ollama and LanceDB, and provides a Pydantic-backed modular representation for all team skills. Each skill has configurable parameters at the top of its cell.

In [ ]:
# Setup and Data Ingestion
import os
import subprocess
import sys
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

# 1. Install necessary dependencies in the environment
def install_packages():
    packages = ["pydantic", "lancedb", "pyarrow", "pandas", "ollama"]
    print("Checking and installing required packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print("Packages installed successfully.")

install_packages()

import lancedb
import pyarrow as pa
import pandas as pd
import ollama

# 2. Check if the repo has been copied, if not copy it
REPO_URL = "https://github.com/jonathanmalkin/jules.git"
REPO_DIR = "jules_repo"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repo from {REPO_URL} into {REPO_DIR}...")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print(f"Repo already exists at {REPO_DIR}.")

# 3. Vectorize it using ollama snowflake into skills/code clusters/parquet and lancedb
EMBEDDING_MODEL = "snowflake-arctic-embed:latest"
PARQUET_OUTPUT_DIR = "skills/code clusters/parquet"
PARQUET_FILE_NAME = "repo_vectors.parquet"
LANCEDB_PATH = "skills/code clusters/lancedb"

print(f"Pulling {EMBEDDING_MODEL} via Ollama...")
try:
    ollama.pull(EMBEDDING_MODEL)
except Exception as e:
    print(f"Error pulling model (make sure Ollama is running): {e}")

documents = []
for root, _, files in os.walk(REPO_DIR):
    for file in files:
        if file.endswith(".md"):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                    documents.append({"filepath": filepath, "content": content})
            except Exception:
                pass

vectorized_data = []
for doc in documents:
    try:
        response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=doc["content"])
        embedding = response["embedding"]
        vectorized_data.append({
            "filepath": doc["filepath"],
            "content": doc["content"][:1000],
            "vector": embedding
        })
    except Exception as e:
        print(f"Failed to embed {doc['filepath']}: {e}")

if vectorized_data:
    df = pd.DataFrame(vectorized_data)
    
    os.makedirs(PARQUET_OUTPUT_DIR, exist_ok=True)
    parquet_path = os.path.join(PARQUET_OUTPUT_DIR, PARQUET_FILE_NAME)
    df.to_parquet(parquet_path)
    print(f"Saved Parquet file to {parquet_path}")
    
    os.makedirs(LANCEDB_PATH, exist_ok=True)
    db = lancedb.connect(LANCEDB_PATH)
    
    if "jules_repo" in db.table_names():
        db.drop_table("jules_repo")
        
    table = db.create_table("jules_repo", data=df)
    print(f"Saved to LanceDB table 'jules_repo' at {LANCEDB_PATH}")
else:
    print("No data vectorized.")


## think
**Description:** Recursive decomposition + advisory. Altitude system for goals, adversarial review for decisions.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for think
THINK_OUTPUT_LOG_PATH = "logs/think_decisions.log"

class ThinkRequest(BaseModel):
    goal: str = Field(..., description="The main objective to decompose.")
    context: Optional[str] = Field(None, description="Advisory context.")

def run_think(req: ThinkRequest):
    print(f"[Think Skill] Analyzing goal: {req.goal}")
    print(f"Writing think logs to: {THINK_OUTPUT_LOG_PATH}")
    return {"status": "success", "altitude": "high", "review": "Adversarial review complete."}


## build
**Description:** Software dev end-to-end: scope, plan, execute, deploy.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for build
BUILD_WORKSPACE_DIR = "workspace/builds"
BUILD_DEPLOY_ENDPOINT = "http://localhost:8080/deploy"

class BuildRequest(BaseModel):
    scope: str = Field(..., description="Scope of the software to build.")
    plan: List[str] = Field(..., description="Steps to execute.")

def run_build(req: BuildRequest):
    print(f"[Build Skill] Building scope: {req.scope} with {len(req.plan)} steps.")
    print(f"Workspace: {BUILD_WORKSPACE_DIR}, Deploying to: {BUILD_DEPLOY_ENDPOINT}")
    return {"status": "success", "deployment": "pending"}


## write
**Description:** Content production: seed to platform-ready output across all channels.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for write
WRITE_OUTPUT_DIR = "content/drafts"
WRITE_PLATFORM_API = "https://api.example.com/publish"

class WriteRequest(BaseModel):
    topic: str = Field(..., description="Topic of the content.")
    platform: str = Field(..., description="Target platform (e.g., blog, X).")

def run_write(req: WriteRequest):
    print(f"[Write Skill] Producing content for {req.platform} on topic: {req.topic}")
    print(f"Saving to: {WRITE_OUTPUT_DIR}, API: {WRITE_PLATFORM_API}")
    return {"status": "success", "output": f"Content for {req.topic}"}


## research
**Description:** Standalone research with persistence and cross-session pickup. Living documents.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for research
RESEARCH_STORAGE_DIR = "research/living_docs"

class ResearchRequest(BaseModel):
    query: str = Field(..., description="Research query.")
    persistence_id: Optional[str] = Field(None, description="ID for cross-session pickup.")

def run_research(req: ResearchRequest):
    print(f"[Research Skill] Researching: {req.query}")
    print(f"Saving living documents to: {RESEARCH_STORAGE_DIR}")
    return {"status": "success", "findings": []}


## debug
**Description:** Systematic debugging: hypothesize, test, narrow.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for debug
DEBUG_LOGS_DIR = "logs/crash_reports"
DEBUG_TOOLS_BIN = "/usr/local/bin/debugger"

class DebugRequest(BaseModel):
    issue_description: str = Field(..., description="Description of the bug.")
    logs: Optional[str] = Field(None, description="Relevant error logs.")

def run_debug(req: DebugRequest):
    print(f"[Debug Skill] Debugging issue: {req.issue_description}")
    print(f"Reading logs from: {DEBUG_LOGS_DIR}")
    return {"status": "success", "hypothesis": "Test required"}


## reply-x
**Description:** Check X mentions, draft replies, post approved ones. Multi-account support.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for reply-x
X_API_ENDPOINT = "https://api.twitter.com/2/tweets"
X_DRAFTS_PATH = "social/x_drafts.json"

class ReplyXRequest(BaseModel):
    mentions: List[str] = Field(..., description="List of X mentions to check.")
    draft_replies: bool = Field(True, description="Whether to draft replies automatically.")

def run_reply_x(req: ReplyXRequest):
    print(f"[Reply-X Skill] Checking {len(req.mentions)} mentions.")
    print(f"API Endpoint: {X_API_ENDPOINT}, Drafts: {X_DRAFTS_PATH}")
    return {"status": "success"}


## good-morning
**Description:** Interactive walkthrough of the morning briefing (10 sections).

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for good-morning
BRIEFING_INPUT_FILE = "briefings/morning_briefing.md"
BRIEFING_SOURCES_DIR = "data/sources"

class GoodMorningRequest(BaseModel):
    include_sections: int = Field(10, description="Number of briefing sections to include.")

def run_good_morning(req: GoodMorningRequest):
    print(f"[Good-Morning Skill] Briefing with {req.include_sections} sections.")
    print(f"Reading briefing from: {BRIEFING_INPUT_FILE}")
    return {"status": "success"}


## wrap-up
**Description:** End-of-session: issue capture, report, ship. 3 phases.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for wrap-up
WRAPUP_REPORT_DIR = "reports/end_of_session"
WRAPUP_ARCHIVE_URL = "https://internal.tools/archive"

class WrapUpRequest(BaseModel):
    phases_completed: int = Field(3, description="Number of end-of-session phases.")
    report_output: str = Field(..., description="Destination for the wrap-up report.")

def run_wrap_up(req: WrapUpRequest):
    print(f"[Wrap-Up Skill] Wrapping up session. Report: {req.report_output}")
    print(f"Saving to: {WRAPUP_REPORT_DIR}, Archiving at: {WRAPUP_ARCHIVE_URL}")
    return {"status": "success"}


## watch-contacts
**Description:** Monitor contacts' X posts, surface engagement opportunities.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for watch-contacts
CONTACTS_LIST_PATH = "data/contacts.csv"
WATCH_POLL_INTERVAL = 600

class WatchContactsRequest(BaseModel):
    contacts: List[str] = Field(..., description="List of contacts to monitor on X.")

def run_watch_contacts(req: WatchContactsRequest):
    print(f"[Watch-Contacts Skill] Monitoring {len(req.contacts)} contacts.")
    print(f"Loading from: {CONTACTS_LIST_PATH}")
    return {"status": "success"}


## simplify-jules
**Description:** On-demand config hygiene — analyzes cold-start bundle for waste and duplication.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for simplify-jules
JULES_BUNDLE_DIR = "config/bundles"
HYGIENE_REPORT_PATH = "logs/hygiene_report.txt"

class SimplifyJulesRequest(BaseModel):
    bundle_path: str = Field(..., description="Path to the cold-start bundle.")

def run_simplify_jules(req: SimplifyJulesRequest):
    print(f"[Simplify-Jules Skill] Auditing bundle at {req.bundle_path}")
    print(f"Default bundle dir: {JULES_BUNDLE_DIR}, Report: {HYGIENE_REPORT_PATH}")
    return {"status": "success"}


## stop-slop
**Description:** Structural audit for AI writing patterns.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for stop-slop
SLOP_DICTIONARY_PATH = "config/slop_words.json"
SLOP_AUDIT_LOG_DIR = "logs/audits"

class StopSlopRequest(BaseModel):
    content_to_audit: str = Field(..., description="Content to audit for AI writing patterns.")

def run_stop_slop(req: StopSlopRequest):
    print("[Stop-Slop Skill] Auditing content for AI patterns.")
    print(f"Using dictionary at: {SLOP_DICTIONARY_PATH}")
    return {"status": "success", "slop_detected": False}


## pdf
**Description:** PDF operations.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for pdf
PDF_WORKSPACE_DIR = "workspace/pdfs"
PDF_EXTRACTOR_BIN = "/usr/bin/pdftotext"

class PdfRequest(BaseModel):
    pdf_path: str = Field(..., description="Path to the PDF file.")
    operation: str = Field(..., description="Operation to perform (e.g., extract text).")

def run_pdf(req: PdfRequest):
    print(f"[PDF Skill] Performing {req.operation} on {req.pdf_path}")
    print(f"Workspace: {PDF_WORKSPACE_DIR}, Tool: {PDF_EXTRACTOR_BIN}")
    return {"status": "success"}


## plane
**Description:** Plane.so interface: MCP tools + gap scripts + reconciliation.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for plane
PLANE_API_URL = "https://api.plane.so/v1"
PLANE_API_KEY_ENV = "PLANE_API_KEY"

class PlaneRequest(BaseModel):
    action: str = Field(..., description="Plane.so interface action.")

def run_plane(req: PlaneRequest):
    print(f"[Plane Skill] Executing action: {req.action}")
    print(f"Using Plane API: {PLANE_API_URL}")
    return {"status": "success"}


## send-email
**Description:** Send email via Resend.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for send-email
RESEND_API_URL = "https://api.resend.com/emails"
RESEND_API_KEY_ENV = "RESEND_API_KEY"

class SendEmailRequest(BaseModel):
    to_address: str = Field(..., description="Recipient email address.")
    subject: str = Field(..., description="Email subject.")
    body: str = Field(..., description="Email body content.")

def run_send_email(req: SendEmailRequest):
    print(f"[Send-Email Skill] Sending email to {req.to_address}")
    print(f"Using Resend API: {RESEND_API_URL}")
    return {"status": "success"}


## financial-advisor
**Description:** Personal finance planning.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for financial-advisor
FINANCE_DATA_DIR = "finance/records"
FINANCE_CALCULATOR_API = "http://localhost:8081/calc"

class FinancialAdvisorRequest(BaseModel):
    planning_topic: str = Field(..., description="Personal finance planning topic.")

def run_financial_advisor(req: FinancialAdvisorRequest):
    print(f"[Financial-Advisor Skill] Planning: {req.planning_topic}")
    print(f"Data directory: {FINANCE_DATA_DIR}")
    return {"status": "success"}


## generate-image-openai
**Description:** Image generation with iterative two-phase workflow.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for generate-image-openai
IMAGE_OUTPUT_DIR = "workspace/images"
OPENAI_IMAGE_API = "https://api.openai.com/v1/images/generations"

class GenerateImageOpenaiRequest(BaseModel):
    prompt: str = Field(..., description="Image generation prompt.")
    iterative_phases: int = Field(2, description="Number of phases for generation workflow.")

def run_generate_image(req: GenerateImageOpenaiRequest):
    print(f"[Image-Gen Skill] Generating image for prompt: {req.prompt}")
    print(f"Saving images to: {IMAGE_OUTPUT_DIR}")
    return {"status": "success"}


## search-history
**Description:** Search session documents.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for search-history
HISTORY_DOCS_DIR = "data/history"
HISTORY_INDEX_PATH = "data/history/index.db"

class SearchHistoryRequest(BaseModel):
    search_term: str = Field(..., description="Term to search in session documents.")

def run_search_history(req: SearchHistoryRequest):
    print(f"[Search-History Skill] Searching for: {req.search_term}")
    print(f"Searching index at: {HISTORY_INDEX_PATH}")
    return {"status": "success"}


## skill-creator
**Description:** Create and modify skills.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for skill-creator
SKILLS_DIR = ".claude/skills"
SKILL_TEMPLATES_DIR = "config/templates"

class SkillCreatorRequest(BaseModel):
    skill_name: str = Field(..., description="Name of the new skill.")
    modifications: Optional[Dict[str, Any]] = Field(None, description="Modifications to an existing skill.")

def run_skill_creator(req: SkillCreatorRequest):
    print(f"[Skill-Creator Skill] Working on skill: {req.skill_name}")
    print(f"Modifying skills in: {SKILLS_DIR}")
    return {"status": "success"}


## agent-browser
**Description:** Browser automation.

*Configure the variables in the code block below to match your environment paths/URLs.*

In [ ]:
# Configuration Options for agent-browser
BROWSER_CDP_URL = "http://localhost:9222"
BROWSER_SCREENSHOTS_DIR = "workspace/screenshots"

class AgentBrowserRequest(BaseModel):
    url: str = Field(..., description="URL to automate browser actions on.")
    actions: List[str] = Field(..., description="List of browser actions to perform.")

def run_agent_browser(req: AgentBrowserRequest):
    print(f"[Agent-Browser Skill] Navigating to {req.url}")
    print(f"Connecting to browser at: {BROWSER_CDP_URL}")
    return {"status": "success"}
